<a href="https://colab.research.google.com/github/yuriserede/An-lise-de-Redes-com-NetworkX/blob/main/Pre-processamento%20de%20Dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# 1. Lê o arquivo CSV original, com suporte a acentos
df = pd.read_csv('clientes_loja.csv', encoding='utf-8')
print("1. Primeiras linhas do dataset:")
print(df.head(), "\n")

# 2. Adiciona um campo de índice único (caso não exista)
df['indice'] = range(len(df))
print("2. Adicionado campo de índice único (coluna 'indice'):")
print(df[['indice']].head(), "\n")

# 3. Corrige valores enganosos (999 para NA) na coluna dias_ultimo_contato
print("3. Quantidade de valores '999' em 'dias_ultimo_contato' antes da substituição:", (df['dias_ultimo_contato'] == 999).sum())
df['dias_ultimo_contato'] = df['dias_ultimo_contato'].replace(999, np.nan)
print("   Quantidade de valores ausentes após a substituição:", df['dias_ultimo_contato'].isna().sum(), "\n")

# 4. Converte variáveis categóricas em numéricas
df['sexo_num'] = df['sexo'].map({'F': 0, 'M': 1})
df['escolaridade_num'] = df['escolaridade'].map({'fundamental': 1, 'médio': 2, 'superior': 3})
df['comprou_num'] = df['comprou'].map({'não': 0, 'sim': 1})
print("4. Exemplos de variáveis categóricas convertidas para numéricas:")
print(df[['sexo', 'sexo_num', 'escolaridade', 'escolaridade_num', 'comprou', 'comprou_num']].head())
print("...\n")

# 5. Padroniza as variáveis numéricas idade e renda_anual (z-score)
df['idade_z'] = stats.zscore(df['idade'])
df['renda_z'] = stats.zscore(df['renda_anual'])
print("5. Idade original vs. z-score:")
print(df[['idade', 'idade_z']].head())
print("...\n")

# 6. Identifica outliers de idade (z-score > 3 ou < -3)
outliers_idade = df[(df['idade_z'] > 3) | (df['idade_z'] < -3)]
print("6. Outliers de idade encontrados:")
print(outliers_idade[['id_cliente', 'idade', 'idade_z']], "\n")

# 6b. (NOVO) Tratar outliers de idade (exemplo: Winsorização)
# Substituir idades > 100 por 100 (motivo: limites biológicos humanos, previne distorção estatística)
df.loc[df['idade'] > 100, 'idade'] = 100
print("6b. Idades acima de 100 anos foram redefinidas para 100 por critério biológico.")

# Recalcular z-score após tratamento dos outliers
df['idade_z'] = stats.zscore(df['idade'])

# 7. Exporta o DataFrame pré-processado para arquivo CSV

# Após todas as transformações e antes de salvar o arquivo:
# Reordena colocando 'indice' como primeira coluna
cols = list(df.columns)
cols.insert(0, cols.pop(cols.index('indice')))
df = df[cols]

# Agora exporta normalmente
df.to_csv('clientes_loja_tratado.csv', sep=',', index=False, encoding='utf-8')
print("   Arquivo 'clientes_loja_tratado.csv' também salvo (formato tradicional CSV).")
print(df.columns)  # Veja todas as colunas do DataFrame


'''
# 7. Exporta o DataFrame pré-processado para arquivo TXT (delimitado por tabulação)
df.to_csv('clientes_loja_tratado.txt', sep='\t', index=False, encoding='utf-8')
print("7. Arquivo pré-processado salvo como 'clientes_loja_tratado.txt' (delimitado por tabulação e com acentuação UTF-8).")
'''

1. Primeiras linhas do dataset:
   id_cliente  idade sexo  renda_anual escolaridade  dias_ultimo_contato  \
0           1     22    F        15000        médio                   12   
1           2     35    M        35000     superior                  999   
2           3     28    F        28000        médio                    5   
3           4     47    M        47000  fundamental                  999   
4           5     19    F        12000        médio                    8   

  comprou  
0     não  
1     sim  
2     não  
3     sim  
4     não   

2. Adicionado campo de índice único (coluna 'indice'):
   indice
0       0
1       1
2       2
3       3
4       4 

3. Quantidade de valores '999' em 'dias_ultimo_contato' antes da substituição: 7
   Quantidade de valores ausentes após a substituição: 7 

4. Exemplos de variáveis categóricas convertidas para numéricas:
  sexo  sexo_num escolaridade  escolaridade_num comprou  comprou_num
0    F         0        médio                 

'\n# 7. Exporta o DataFrame pré-processado para arquivo TXT (delimitado por tabulação)\ndf.to_csv(\'clientes_loja_tratado.txt\', sep=\'\t\', index=False, encoding=\'utf-8\')\nprint("7. Arquivo pré-processado salvo como \'clientes_loja_tratado.txt\' (delimitado por tabulação e com acentuação UTF-8).")\n'